In [0]:
# Cell 1: Mocking the Raw Extract Ingestion
from pyspark.sql.functions import col, when, upper, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

print("Initializing Ingestion Handshake from ADF...")

# Since we are using Serverless Free Edition, let's mock a data payload to test the pipeline run
mock_data = [
    (1, 0, 3, "Braund, Mr. Owen Harris", "male", 22.0),
    (2, 1, 1, "Cumings, Mrs. John Bradley (Florence Briggs Thayer)", "female", 38.0),
    (3, 1, 3, "Heikkinen, Miss. Laina", "female", 26.0),
    (None, None, None, None, None, None) # Dirty null row to test our filter
]

schema = StructType([
    StructField("PassengerId", IntegerType(), True),
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", StringType(), True)
])

df_raw = spark.createDataFrame(mock_data, schema)
print(f"Raw mock row count: {df_raw.count()}")
df_raw.printSchema()

Initializing Ingestion Handshake from ADF...
Raw mock row count: 4
root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)



In [0]:
# Basic Cleaning
df_clean = df_raw \
    .withColumn("Name", trim(col("Name"))) \
    .withColumn("Sex", upper(col("Sex"))) \
    .withColumn("Survived", col("Survived").cast("integer")) \
    .withColumn("Age", col("Age").cast("double")) \
    .filter(col("PassengerId").isNotNull())

print(f"Clean row count: {df_clean.count()}")
df_clean.show(5)

Clean row count: 3
+-----------+--------+------+--------------------+------+----+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|
+-----------+--------+------+--------------------+------+----+
|          1|       0|     3|Braund, Mr. Owen ...|  MALE|22.0|
|          2|       1|     1|Cumings, Mrs. Joh...|FEMALE|38.0|
|          3|       1|     3|Heikkinen, Miss. ...|FEMALE|26.0|
+-----------+--------+------+--------------------+------+----+

